# M5 — Baseline V3 Offline Verification

**Purpose:** Reproduce local tests and independently audit the **preserved** 20-case × 3-seed Baseline V3 evidence. No live Gemini generation is performed.

**Inputs to upload when prompted:** `riyadh-travel-planner-m5-main.zip` (project snapshot), `riyadh_m4_datasets.zip` (original three datasets), and historical `evaluate.txt` and `experiment.txt` from the earlier GitHub commit. The two historical source files must match the frozen SHA-256 hashes. Keep the original notebook as an unedited audit trail.

**Note:** Output cells retained below are from the previous successful Colab run; they are **not** a new execution of this cleaned notebook. The notebook must be run in a fresh Colab session to reproduce them. The current repository files are not overwritten.

## 1. Upload and extract project and datasets

In [5]:

from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import hashlib

root = Path('/content')
project = root / 'riyadh-travel-planner-m5-main'

uploaded = {}

print('Upload the GitHub project ZIP and original dataset ZIP.')
uploaded = files.upload()

project_zip = next(
    (root / name for name in uploaded
     if name.startswith('riyadh-travel-planner-m5-main') and name.endswith('.zip')),
    None
)
datasets_zip = next(
    (root / name for name in uploaded
     if name.startswith('riyadh_m4_datasets') and name.endswith('.zip')),
    None
)

assert project_zip is not None, 'Project ZIP missing.'
assert datasets_zip is not None, 'Dataset ZIP missing.'
assert not project.exists(), 'Project folder already exists; use a fresh Colab runtime.'

with ZipFile(project_zip) as z:
    z.extractall(root)

assert project.is_dir(), 'Expected project directory was not extracted.'

expected = {
    'data/activities.csv': 'df5abb1fbb5fe13e0e8f20776fc83f707f2d7bf5e610a36acedb06cffa76d01f',
    'data/restaurants.csv': '2f808d6007d026b3c11b38ea9c922560b97920f3da82020b3662b24313c63cbc',
    'data/hotels.csv': '910e7b6055b37fd76c0aec254e4e8f5cf9a6028735096a112f512f3c715ae878',
}

with ZipFile(datasets_zip) as z:
    for name, expected_hash in expected.items():
        data = z.read(name)
        assert hashlib.sha256(data).hexdigest() == expected_hash, f'Dataset hash mismatch: {name}'
        destination = project / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)
        print('VERIFIED:', name)

print('PROJECT AND DATASETS READY')


Upload the GitHub project ZIP and original dataset ZIP.


Saving riyadh_m4_datasets.zip to riyadh_m4_datasets (2).zip
Saving riyadh-travel-planner-m5-main (1).zip to riyadh-travel-planner-m5-main (1) (2).zip
VERIFIED: data/activities.csv
VERIFIED: data/restaurants.csv
VERIFIED: data/hotels.csv
PROJECT AND DATASETS READY


## 2. Install the frozen SDK dependencies

In [6]:
import subprocess, sys
from importlib.metadata import version
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project / 'requirements-gemini.txt')], check=True)
for pkg in ('google-genai', 'httpx', 'pydantic'):
    print(pkg, version(pkg))

google-genai 1.30.0
httpx 0.28.1
pydantic 2.14.3


## 3. Run automated offline tests

In [7]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=project, capture_output=True, text=True, timeout=180)
print('EXIT CODE:', result.returncode)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'Offline test suite failed.'

EXIT CODE: 0

test_activity_exclusion_and_zero_candidates (test_components.ComponentTests.test_activity_exclusion_and_zero_candidates) ... ok
test_activity_search_respects_limit_and_interests (test_components.ComponentTests.test_activity_search_respects_limit_and_interests) ... ok
test_budget_exclusion (test_components.ComponentTests.test_budget_exclusion) ... ok
test_hotel_search_respects_budget_and_limit (test_components.ComponentTests.test_hotel_search_respects_budget_and_limit) ... ok
test_invalid_tool_inputs (test_components.ComponentTests.test_invalid_tool_inputs) ... ok
test_original_dataset_file_hashes (test_components.ComponentTests.test_original_dataset_file_hashes)
Verify original CSV bytes, not reconstructed DataFrame hashes. ... ok
test_restaurant_search_respects_budget_and_limit (test_components.ComponentTests.test_restaurant_search_respects_budget_and_limit) ... ok
test_search_results_are_deterministic (test_components.ComponentTests.test_search_results_are_deterministic

## 4. Upload and verify historical source files

The archived V3 manifest records the original source hashes. The current GitHub versions of `evaluate.py` and `experiment.py` differ; historical copies are required for the **isolated** audit. Upload them as `evaluate.txt` and `experiment.txt` (Colab may add a suffix such as `(1)`).

In [8]:
from pathlib import Path
from google.colab import files
import hashlib, shutil

historical = {
    'evaluate': '783d38e783060d585bce8f6d7cfd2eec896289597c082ae964c84baab02c283f',
    'experiment': 'f5783fce81b04f39cc34a03b54853dcc1a20a3f4618775c433676ba5c5f022f9',
}
print('Upload the two historical text files.')
uploaded = files.upload()
verified = {}
for name, expected_hash in historical.items():
    candidates = [(filename, content) for filename, content in uploaded.items() if filename.startswith(name) and filename.endswith(('.txt', '.py'))]
    assert len(candidates) == 1, f'Expected one uploaded file for {name}'
    filename, content = candidates[0]
    assert hashlib.sha256(content).hexdigest() == expected_hash, f'Historical source hash mismatch: {filename}'
    verified[name] = content
    print('HISTORICAL HASH VERIFIED:', name)

verification = Path('/content/m5-baseline-verification')
assert not verification.exists(), 'Verification folder exists; use a fresh runtime.'
shutil.copytree(project, verification)
for name, content in verified.items():
    (verification / 'm5eval' / f'{name}.py').write_bytes(content)
print('ISOLATED COPY CREATED; ORIGINAL PROJECT UNCHANGED')

Upload the two historical text files.


Saving evaluate.txt to evaluate.txt
Saving experiment.txt to experiment.txt
HISTORICAL HASH VERIFIED: evaluate
HISTORICAL HASH VERIFIED: experiment
ISOLATED COPY CREATED; ORIGINAL PROJECT UNCHANGED


## 5. Independently audit the saved 60-run cohort

In [9]:
import json
result = subprocess.run([sys.executable, 'scripts/verify_full_cohort.py'], cwd=verification, capture_output=True, text=True, timeout=180)
print('EXIT CODE:', result.returncode)
try:
    report = json.loads(result.stdout)
    for key in ('status', 'expected_records', 'verified_records', 'generation_successes', 'structurally_valid_runs', 'saved_summary_verified', 'source_files_matching', 'datasets_matching'):
        print(f'{key}: {report.get(key)}')
    if report.get('status') == 'FAIL':
        print('ERROR:', report.get('message'))
except json.JSONDecodeError:
    print(result.stdout[-4000:])
if result.stderr:
    print('STDERR:', result.stderr[-3000:])
assert result.returncode == 0 and report.get('status') == 'PASS_EVIDENCE_AUDIT', 'Evidence audit did not pass.'

EXIT CODE: 0
status: PASS_EVIDENCE_AUDIT
expected_records: 60
verified_records: 60
generation_successes: 58
structurally_valid_runs: 49
saved_summary_verified: True
source_files_matching: 13
datasets_matching: 3


## Interpretation and limitations

The prior successful audit returned `PASS_EVIDENCE_AUDIT`, verified **60/60 saved runs**, **58 successful generations**, **49 structurally valid runs**, **13/13 historical source fingerprints**, **3/3 dataset fingerprints**, and the saved summary. This confirms saved-evidence integrity and offline rescoring **not** 100% itinerary quality, a fresh provider run, or provider-side attestation.

The notebook requires external ZIPs and historical source files; these are not embedded in the `.ipynb`. The original historical run notebook is retained separately.